# Genie Benchmark Runner

This notebook runs a Genie space against its configured benchmark questions across **N independent runs**
and records accuracy results in MLflow.

### What it does

1. Pulls the Genie space configuration (benchmark questions + expected SQL) by space ID
2. Runs every benchmark question through the live Genie API **NUM_RUNS** times
3. Uses an LLM to evaluate whether each generated query is semantically equivalent to the expected SQL
4. Outputs a results DataFrame with: run index, question, generated SQL, and pass/fail
5. Creates an MLflow run logging overall accuracy (avg across all runs) and per-run accuracy

### Pass / Fail Logic

| Condition | Result |
|-----------|--------|
| Benchmark has expected SQL and generated SQL matches semantically | PASS |
| Benchmark has expected SQL but generated SQL does not match | FAIL |
| Benchmark has no expected SQL and a SQL was generated | PASS |
| No SQL was generated (error or empty response) | FAIL |

In [ ]:
%pip install databricks-sdk langchain langchain-databricks langchain-core mlflow typing-extensions -qU

dbutils.library.restartPython()

In [ ]:
import json
import time
import pandas as pd
from datetime import datetime

import mlflow
from databricks.sdk import WorkspaceClient
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_databricks import ChatDatabricks

## Configuration

Set your Genie space ID, LLM endpoint, and number of benchmark runs below.

When running inside a Databricks notebook, `WorkspaceClient()` authenticates automatically.
For local execution, set the `DATABRICKS_HOST` and `DATABRICKS_TOKEN` environment variables.

In [ ]:
GENIE_SPACE_ID  = "<your-genie-space-id>"
LLM_ENDPOINT    = "databricks-meta-llama-3-1-70b-instruct"
NUM_RUNS        = 5    # number of complete benchmark passes to execute
MAX_WAIT_SEC    = 180  # per-question timeout in seconds
MLFLOW_EXPERIMENT_NAME = "/genie-benchmark-runner"

## Load Genie Space & Benchmark Questions

Fetch the full space definition from the Databricks Genie API. Benchmark questions live
under `benchmarks.questions` in the serialized space config. Each question may include
an `sql` field containing the expected query — this is what we evaluate generated output against.

In [ ]:
w = WorkspaceClient()

space = w.genie.get_space(
    space_id=GENIE_SPACE_ID,
    include_serialized_space=True,
)

space_config       = json.loads(space.serialized_space)
benchmark_questions = space_config.get("benchmarks", {}).get("questions", [])

# Normalise each benchmark to {question: str, expected_sql: str | None}
benchmarks = []
for q in benchmark_questions:
    question_text = " ".join(q.get("question", [])).strip()
    expected_sql = None
    for answer in q.get("answer", []):
        if answer.get("format") == "SQL" and answer.get("content"):
            expected_sql = "".join(answer["content"]).strip()
            break
    if question_text:
        benchmarks.append({"question": question_text, "expected_sql": expected_sql})

print(f"Space:               {space.title}")
print(f"Description:         {space.description or 'N/A'}")
print(f"Benchmark questions: {len(benchmarks)}")
print(f"Runs planned:        {NUM_RUNS}")
print(f"Total API calls:     {len(benchmarks) * NUM_RUNS}")

print("\n--- Benchmark Questions ---")
for i, b in enumerate(benchmarks, 1):
    print(f"  {i:>2}. {b['question']}")
    if b["expected_sql"]:
        for line in b["expected_sql"].splitlines():
            print(f"        {line}")
    else:
        print(f"        (no expected SQL)")

## Helper Functions

### `run_genie_question`
Starts a new Genie conversation for a question and polls until the response is complete,
then extracts the generated SQL from the message attachments.

### `evaluate_pass`
Uses the LLM to determine whether the generated SQL is semantically equivalent to the
expected SQL. Semantic equivalence means the two queries would return the same result
set on the same data — minor aliasing or formatting differences are acceptable.

In [ ]:
def run_genie_question(w, space_id, question_text, max_wait_seconds=MAX_WAIT_SEC):
    """
    Submit a question to a Genie space and return (generated_sql, error_message).

    Returns
    -------
    generated_sql : str | None
        The SQL query generated by Genie, or None on failure.
    error : str | None
        A description of what went wrong, or None on success.
    """
    try:
        # Start a new conversation
        resp = w.genie.start_conversation(space_id=space_id, content=question_text)

        # Extract conversation / message IDs — handle both flat and nested SDK shapes
        if hasattr(resp, "conversation_id") and hasattr(resp, "message_id"):
            conv_id = resp.conversation_id
            msg_id  = resp.message_id
        elif hasattr(resp, "conversation") and hasattr(resp, "message"):
            conv_id = resp.conversation.id
            msg_id  = resp.message.id
        else:
            return None, f"Unexpected start_conversation response shape: {type(resp)}"

        # Poll until the message reaches a terminal status
        deadline = time.time() + max_wait_seconds
        msg = None
        while time.time() < deadline:
            msg = w.genie.get_message(
                space_id=space_id,
                conversation_id=conv_id,
                message_id=msg_id,
            )
            status = msg.status.value if hasattr(msg.status, "value") else str(msg.status)
            status_upper = status.upper()

            if status_upper == "COMPLETED":
                # Pull SQL from attachments
                for att in (getattr(msg, "attachments", None) or []):
                    query_part = getattr(att, "query", None)
                    if query_part:
                        sql = getattr(query_part, "query", None)
                        if sql:
                            return sql.strip(), None
                return None, "Completed but no SQL found in message attachments"

            elif status_upper in ("FAILED", "ERROR", "CANCELLED", "QUERY_RESULT_EXPIRED"):
                return None, f"Message ended with status: {status}"

            time.sleep(3)

        return None, f"Timeout after {max_wait_seconds}s — last status: {status}"

    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"

In [ ]:
llm = ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.0)

_eval_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert SQL evaluator. Your job is to determine whether a generated SQL "
     "query is semantically equivalent to an expected SQL query. Two queries are semantically "
     "equivalent if they would produce the same result set on the same data. Minor differences "
     "in column aliasing, whitespace, or capitalisation are acceptable. However, differences "
     "in filters, aggregations, joins, or selected columns that would change the result are NOT "
     "acceptable. Respond with valid JSON only — no markdown fences."),
    ("human",
     """Question: {question}

Expected SQL:
{expected_sql}

Generated SQL:
{generated_sql}

Determine whether the generated SQL is semantically equivalent to the expected SQL.

Respond with ONLY this JSON:
{{"passed": true_or_false, "reasoning": "<one sentence>"}}"""),
])

_eval_chain = _eval_prompt | llm | JsonOutputParser()


def evaluate_pass(question, expected_sql, generated_sql):
    """
    Determine whether a generated SQL passes the benchmark.

    - With expected SQL  : LLM semantic equivalence check
    - Without expected SQL: passes if any SQL was generated

    Returns
    -------
    passed   : bool
    reasoning : str
    """
    if not generated_sql:
        return False, "No SQL was generated"

    if not expected_sql:
        return True, "SQL generated successfully (no expected SQL to compare against)"

    try:
        result = _eval_chain.invoke({
            "question":      question,
            "expected_sql":  expected_sql,
            "generated_sql": generated_sql,
        })
        return bool(result.get("passed", False)), result.get("reasoning", "")
    except Exception as exc:
        # Fallback: exact string match
        exact = expected_sql.strip().lower() == generated_sql.strip().lower()
        return exact, f"LLM evaluation failed ({exc}); fell back to exact match: {exact}"

## Run Benchmark Executions

Execute all benchmark questions **NUM_RUNS** times. Each run sends every question to the
live Genie API, collects the generated SQL, and evaluates pass/fail.

> **Note:** With 20 benchmark questions and 5 runs, this makes 100 Genie API calls.
> Allow ~3–10 minutes depending on space complexity and cluster warm-up time.

In [ ]:
if not benchmarks:
    raise ValueError(
        f"No benchmark questions found in space '{GENIE_SPACE_ID}'. "
        "Add benchmark questions to the Genie space before running this notebook."
    )

raw_results = []  # list of dicts, one per (run, question)

for run_idx in range(1, NUM_RUNS + 1):
    print(f"\n{'='*60}")
    print(f"  Run {run_idx} / {NUM_RUNS}")
    print(f"{'='*60}")

    run_pass  = 0
    run_total = len(benchmarks)

    for q_idx, benchmark in enumerate(benchmarks, 1):
        question     = benchmark["question"]
        expected_sql = benchmark["expected_sql"]

        print(f"  [{run_idx}/{NUM_RUNS}] Q{q_idx:>2}/{run_total} — {question[:80]}..." if len(question) > 80
              else f"  [{run_idx}/{NUM_RUNS}] Q{q_idx:>2}/{run_total} — {question}")

        # Step 1: ask Genie
        generated_sql, genie_error = run_genie_question(w, GENIE_SPACE_ID, question)

        # Step 2: evaluate
        passed, reasoning = evaluate_pass(question, expected_sql, generated_sql)

        status_icon = "PASS" if passed else "FAIL"
        print(f"          -> [{status_icon}] {reasoning[:100]}")
        if genie_error:
            print(f"          -> [ERROR] {genie_error}")

        if passed:
            run_pass += 1

        raw_results.append({
            "run_index":    run_idx,
            "question":     question,
            "expected_sql": expected_sql,
            "generated_sql": generated_sql,
            "passed":       passed,
            "reasoning":    reasoning,
            "error":        genie_error,
        })

    run_pct = run_pass / run_total * 100
    print(f"\n  Run {run_idx} accuracy: {run_pass}/{run_total}  ({run_pct:.1f}%)")

print("\nAll runs complete.")

## Results DataFrame

Full results table with one row per (run, question). Columns:

| Column | Description |
|--------|-------------|
| `run_index` | Which run (1–NUM_RUNS) |
| `question` | The benchmark question text |
| `expected_sql` | Expected SQL from the Genie space config |
| `generated_sql` | SQL produced by the live Genie API |
| `passed` | Whether the generated SQL passed evaluation |
| `reasoning` | LLM explanation of the pass/fail decision |
| `error` | Any API or evaluation error (None on success) |

In [ ]:
results_df = pd.DataFrame(raw_results)

# Reorder columns for readability
results_df = results_df[[
    "run_index", "question", "expected_sql", "generated_sql",
    "passed", "reasoning", "error",
]]

print(f"Total rows: {len(results_df)}  ({NUM_RUNS} runs × {len(benchmarks)} questions)")

# Display in Databricks notebook (falls back to standard print outside Databricks)
try:
    display(results_df)  # noqa: F821 — display() is a Databricks built-in
except NameError:
    print(results_df.to_string())

## Summary Statistics

Per-run accuracy and overall accuracy averaged across all runs.

In [ ]:
# Per-run accuracy
per_run_accuracy = (
    results_df.groupby("run_index")["passed"]
    .agg(passed_count="sum", total="count")
    .assign(accuracy_pct=lambda df: df["passed_count"] / df["total"] * 100)
    .reset_index()
)

overall_accuracy = per_run_accuracy["accuracy_pct"].mean()

print(f"Space:            {space.title}")
print(f"Benchmark count:  {len(benchmarks)}")
print(f"Runs:             {NUM_RUNS}")
print()
print("Per-run accuracy:")
for _, row in per_run_accuracy.iterrows():
    bar = '█' * int(row['accuracy_pct'] / 5) + '░' * (20 - int(row['accuracy_pct'] / 5))
    print(f"  Run {int(row['run_index'])}: {bar} {row['accuracy_pct']:5.1f}%  "
          f"({int(row['passed_count'])}/{int(row['total'])})")

bar = '█' * int(overall_accuracy / 5) + '░' * (20 - int(overall_accuracy / 5))
print(f"\n  Overall: {bar} {overall_accuracy:5.1f}%")

try:
    display(per_run_accuracy)  # noqa: F821
except NameError:
    print(per_run_accuracy.to_string())

## MLflow Run

Log all results to MLflow:

| Logged Item | Type | Description |
|-------------|------|-------------|
| `space_id` | param | Genie space ID |
| `space_name` | param | Human-readable space name |
| `num_benchmarks` | param | Number of benchmark questions |
| `num_runs` | param | Number of runs executed |
| `overall_accuracy` | metric | Average accuracy across all runs (%) |
| `run_N_accuracy` | metric | Per-run accuracy (%) for runs 1–N |
| `benchmark_results.csv` | artifact | Full results DataFrame |

In [ ]:
import tempfile
import os

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"{space.title.replace(' ', '_')}_{run_timestamp}"

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name=run_name) as mlflow_run:

    # --- Parameters ---
    mlflow.log_params({
        "space_id":       GENIE_SPACE_ID,
        "space_name":     space.title,
        "num_benchmarks": len(benchmarks),
        "num_runs":       NUM_RUNS,
        "llm_endpoint":   LLM_ENDPOINT,
    })

    # --- Per-run accuracy metrics ---
    for _, row in per_run_accuracy.iterrows():
        run_num = int(row["run_index"])
        mlflow.log_metric(f"run_{run_num}_accuracy", round(row["accuracy_pct"], 2))

    # --- Overall accuracy ---
    mlflow.log_metric("overall_accuracy", round(overall_accuracy, 2))

    # --- Save full results as CSV artifact ---
    with tempfile.TemporaryDirectory() as tmpdir:
        csv_path = os.path.join(tmpdir, "benchmark_results.csv")
        results_df.to_csv(csv_path, index=False)
        mlflow.log_artifact(csv_path)

    run_id  = mlflow_run.info.run_id
    run_url = mlflow_run.info.run_id  # replace with tracking URL if needed

print(f"MLflow run complete.")
print(f"  Run name:         {run_name}")
print(f"  Run ID:           {run_id}")
print(f"  Experiment:       {MLFLOW_EXPERIMENT_NAME}")
print(f"  Overall accuracy: {overall_accuracy:.1f}%")
print()
print("Per-run metrics logged:")
for _, row in per_run_accuracy.iterrows():
    print(f"  run_{int(row['run_index'])}_accuracy = {row['accuracy_pct']:.2f}%")
print(f"  overall_accuracy = {overall_accuracy:.2f}%")